In [ ]:
import os  # added 2026: credentials were scrubbed to environment reads
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import pandas_ta as ta
from sklearn.impute import SimpleImputer
from datetime import datetime, timezone, timedelta
import time
import pytz
from threading import Event
import joblib
import sys

def get_signal():
    ticker = 'XAUUSD_i'
    interval = mt5.TIMEFRAME_M15
    rates = mt5.copy_rates_from_pos(ticker, interval, 1, 55)
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.rename(columns={
        'time': 'Date',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close'
    }, inplace=True)
    df.set_index('Date', inplace=True)
    df = df.drop(columns=['tick_volume', 'real_volume', 'spread'])
    df[["lowerBB2", "midBB","upperBB2","bandwidthBB2","percentBB2"]] = ta.bbands(df["Close"], length=21, std=2)
    df["returns"] = np.log(df.Close.div(df.Close.shift(1)))
    df.dropna(inplace=True)
    df["min_6"] = df['Close'].rolling(6).min() / df['Close'] - 1
    df["max_6"] = df['Close'].rolling(6).max() / df['Close'] - 1
    df["boll_6"] = (df['Close'] - df['Close'].rolling(6).mean()) / df['Close'].rolling(6).std()
    df["min"] = df['Close'].rolling(30).min() / df['Close'] - 1
    df["max"] = df['Close'].rolling(30).max() / df['Close'] - 1
    df["boll"] = (df['Close'] - df['Close'].rolling(30).mean()) / df['Close'].rolling(30).std()
    df['min_6 / min'] = df['min_6'] / df['min']
    df['max_6 / max'] = df['max_6'] / df['max']
    df['min / max'] = df['min'] / df['max']
    df['min_6 / max_6'] = df['min_6'] / df['max_6']
    df['boll_6 / boll'] = df['boll_6'] / df['boll']
        
    # Replace infinities and handle NaNs
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Ensure columns exist and impute missing values
    columns_to_impute = ['min_6 / max_6', 'min / max', 'boll_6 / boll']
    imputer = SimpleImputer(strategy='mean')
    for col in columns_to_impute:
        if col in df.columns:
            df[[col]] = imputer.fit_transform(df[[col]])

    # Final cleanup (if needed)
    df.dropna(inplace=True)
    
    X = df.drop(columns=['Open','High','Low','Close',"lowerBB2", "midBB","upperBB2","bandwidthBB2"])
    phrase = '_lag_1'
    X.columns = [col + phrase for col in X.columns]
    
    model = joblib.load('U.joblib')
    signal = model.predict(X)
    
    return signal[-1]

def get_open_position():
    positions = mt5.positions_get()
    if positions:
        return positions[0]  # Assuming only one position for simplicity
    return None

def close_position(position):
    ticket = position.ticket
    if position.type == mt5.ORDER_TYPE_BUY:
        close_action = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(position.symbol).bid
    else:
        close_action = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(position.symbol).ask
    
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": position.symbol,
        "volume": position.volume,
        "type": close_action,
        "position": ticket,
        "price": price,
        "comment": "close the position",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    
    result = mt5.order_send(request)
    print(f"Close order result: {result}")


def execute_trade(signal, qty):
    ticker = 'XAUUSD_i'
    if signal == 1:
        order_type = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(ticker).ask
        action = "BUY"
    elif signal == -1:
        order_type = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(ticker).bid
        action = "SELL"
    else:
        print("No action needed")
        return
    
    print(f"Executing {action} order: {ticker}, Volume: {qty}, Price: {price}")
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": ticker,
        "volume": qty,
        "type": order_type,
        "price": price,
        "comment": "python open",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    result = mt5.order_send(request)
    print(f"Trade order result: {result}")

def get_nyc_time():
    tz_nyc = pytz.timezone('America/New_York')
    now = datetime.now(tz_nyc)
    return now.strftime('%Y-%m-%d %H:%M:%S')
    
# Define the global variable
minutes = 0

def get_next_bar_time(interval):
    now = datetime.now()  # Use local time for simplicity
    if interval == mt5.TIMEFRAME_M15:
        minutes_past = now.minute % 15
        minutes_to_next_bar = (15 - minutes_past) % 15
        if minutes_to_next_bar == 0:
            minutes_to_next_bar = 15
        
        # Calculate the exact next bar time
        next_bar_time = now.replace(second=0, microsecond=0) + timedelta(minutes=minutes_to_next_bar)
        return next_bar_time
    else:
        raise ValueError("Unsupported timeframe")


def check_and_trade(stop_event):
    last_signal = None

    while not stop_event.is_set():
        try:
            # Print NYC time and waiting time
            NYC = get_nyc_time()
            print(f"Current NYC time: {NYC}")
            
            # Fetch current signal
            current_signal = get_signal()
            print(f"New signal checked: {current_signal}")

            # Fetch open position
            open_position = get_open_position()

            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}.previous was opposite closing position.")
                    close_position(open_position)
                    # After closing, wait to ensure the position is closed before opening a new one
                    time.sleep(3)
                    open_position = get_open_position()  # Re-fetch the open position status
                    if open_position is None:
                        execute_trade(current_signal, 1.00)
                        print(f"------------------------------------------------------------------------")
                    else:
                        print("Failed to close the position. Not executing new trade.")
                        print(f"------------------------------------------------------------------------")
                else:
                    print(f"Position exists but the signal is the same or mismatched. No action needed.")
                    print(f"------------------------------------------------------------------------")
            else:
                if current_signal:
                    execute_trade(current_signal, 1.00)
                else:
                    print("No signal to act upon.")
                    print(f"------------------------------------------------------------------------")

            # Update last_signal
            last_signal = current_signal

            # Get next bar time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)
            next_check_time = next_bar_time + timedelta(seconds=3)

            # Wait until 3 seconds after the bar closes
            while datetime.now() < next_check_time:
                time.sleep(0.1)  # Sleep briefly to avoid busy waiting

        except Exception as e:
            print(f"An error occurred: {e}")
            time.sleep(60)  # Wait before retrying in case of error



# Initialize MetaTrader 5 connection and login
mt5.initialize()
username = int(os.environ['MT5_LOGIN'])
password = os.environ['MT5_PASSWORD']
server = 'Alpari-MT5-Demo'
mt5.login(username, password, server)

# Create a stop event
stop_event = Event()

try:
    # Start the trading loop
    check_and_trade(stop_event)
except KeyboardInterrupt:
    # Handle manual interruption
    print("Interrupted by user")
finally:
    # Shutdown MetaTrader 5 connection when done
    mt5.shutdown()
    print("MetaTrader 5 connection closed")
